In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import os
import zipfile
from PIL import Image
import urllib.request
import random

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
IMG_SIZE    = 128       # Reduced from 256 → 4x faster
BATCH_SIZE  = 32        # Reduced from 16 (128x128 is lighter)
NUM_EPOCHS  = 30        # Reduced from 50
LR          = 0.0002
BETA1       = 0.5
BETA2       = 0.999
L1_LAMBDA   = 100
NUM_WORKERS = 2

print(f'Image Size : {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch Size : {BATCH_SIZE}')
print(f'Epochs     : {NUM_EPOCHS}')

In [ ]:
import subprocess

# Download edges2shoes dataset via PyTorch's pix2pix dataset script
url = "http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/edges2shoes.tar.gz"

if not os.path.exists('./edges2shoes'):
    print("Downloading edges2shoes dataset (~2GB)...")
    subprocess.run(['wget', '-q', '--show-progress', '-O', 'edges2shoes.tar.gz', url])
    print("Extracting...")
    subprocess.run(['tar', '-xzf', 'edges2shoes.tar.gz'])
    print("Done!")
else:
    print("Dataset already exists.")

# Check structure
train_dir = './edges2shoes/train'
val_dir   = './edges2shoes/val'
train_files = os.listdir(train_dir)
val_files   = os.listdir(val_dir)
print(f'Train images: {len(train_files)}')
print(f'Val   images: {len(val_files)}')

# Preview one image to understand format
sample_path = os.path.join(train_dir, train_files[0])
sample = Image.open(sample_path)
print(f'Sample image size: {sample.size}')  # Should be 512x256 (side-by-side pair)
plt.figure(figsize=(8, 4))
plt.imshow(sample)
plt.title('Sample: Left=Edge, Right=Real Shoe')
plt.axis('off')
plt.show()

In [ ]:
from torch.utils.data import DataLoader, Dataset, Subset

class Edges2ShoesDataset(Dataset):
    def __init__(self, root_dir, split='train', img_size=128, max_samples=None):
        self.root_dir = os.path.join(root_dir, split)
        self.files    = sorted(os.listdir(self.root_dir))

        # Limit samples directly here — no Subset needed
        if max_samples is not None:
            self.files = self.files[:max_samples]

        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.files[idx])
        img      = Image.open(img_path).convert('RGB')
        w, h     = img.size
        half_w   = w // 2
        edge_img = img.crop((0, 0, half_w, h))
        real_img = img.crop((half_w, 0, w, h))
        return self.transform(edge_img), self.transform(real_img)


# Load with built-in limit
train_dataset = Edges2ShoesDataset('./edges2shoes', split='train',
                                    img_size=IMG_SIZE, max_samples=3000)
val_dataset   = Edges2ShoesDataset('./edges2shoes', split='val',
                                    img_size=IMG_SIZE, max_samples=300)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train samples : {len(train_dataset)}')
print(f'Val   samples : {len(val_dataset)}')
print(f'Train batches : {len(train_loader)}')

# Quick sanity check
e, r = train_dataset[0]
print(f'Edge shape: {e.shape}, Real shape: {r.shape}')
print(f'Value range: [{e.min():.2f}, {e.max():.2f}]')

In [ ]:
def denormalize(tensor):
    return (tensor * 0.5 + 0.5).clamp(0, 1)

edges, reals = next(iter(val_loader))

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
fig.suptitle('Edges2Shoes Paired Dataset\nTop: Edge Map (Input) | Bottom: Real Shoe (Target)',
             fontsize=12, fontweight='bold')

for i in range(8):
    axes[0, i].imshow(denormalize(edges[i]).permute(1,2,0).numpy())
    axes[0, i].axis('off')
    axes[1, i].imshow(denormalize(reals[i]).permute(1,2,0).numpy())
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Edge Input', fontsize=9)
axes[1, 0].set_ylabel('Real Target', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
class UNetDown(nn.Module):
    def __init__(self, in_ch, out_ch, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, stride=2, padding=1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNetUp(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        ]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x, skip):
        x = self.block(x)
        return torch.cat([x, skip], dim=1)


class UNetGenerator(nn.Module):
    """
    U-Net for 128x128 images — 6 down/up levels (no 7th level that caused size crash).
    128 → 64 → 32 → 16 → 8 → 4 → 2 (bottleneck)
    """
    def __init__(self, in_channels=3, out_channels=3, features=64):
        super().__init__()
        f = features

        # Encoder (6 levels)
        self.down1 = UNetDown(in_channels, f,    normalize=False)  # 128→64
        self.down2 = UNetDown(f,           f*2)                    # 64→32
        self.down3 = UNetDown(f*2,         f*4)                    # 32→16
        self.down4 = UNetDown(f*4,         f*8)                    # 16→8
        self.down5 = UNetDown(f*8,         f*8)                    # 8→4
        self.down6 = UNetDown(f*8,         f*8)                    # 4→2

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(f*8, f*8, 4, stride=2, padding=1),           # 2→1
            nn.ReLU(inplace=True)
        )

        # Decoder (6 levels, skip connections double channels)
        self.up1 = UNetUp(f*8,   f*8, dropout=0.5)   # 1→2
        self.up2 = UNetUp(f*8*2, f*8, dropout=0.5)   # 2→4
        self.up3 = UNetUp(f*8*2, f*8, dropout=0.5)   # 4→8
        self.up4 = UNetUp(f*8*2, f*4)                # 8→16
        self.up5 = UNetUp(f*4*2, f*2)                # 16→32
        self.up6 = UNetUp(f*2*2, f)                  # 32→64

        # Output layer
        self.final = nn.Sequential(
            nn.ConvTranspose2d(f*2, out_channels, 4, stride=2, padding=1),  # 64→128
            nn.Tanh()
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.normal_(m.weight, 0.0, 0.02)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight, 1.0, 0.02)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        d1 = self.down1(x)    # (B,  64, 64, 64)
        d2 = self.down2(d1)   # (B, 128, 32, 32)
        d3 = self.down3(d2)   # (B, 256, 16, 16)
        d4 = self.down4(d3)   # (B, 512,  8,  8)
        d5 = self.down5(d4)   # (B, 512,  4,  4)
        d6 = self.down6(d5)   # (B, 512,  2,  2)
        bn = self.bottleneck(d6)  # (B, 512, 1, 1)

        u1 = self.up1(bn, d6)  # (B,1024,  2,  2)
        u2 = self.up2(u1, d5)  # (B,1024,  4,  4)
        u3 = self.up3(u2, d4)  # (B,1024,  8,  8)
        u4 = self.up4(u3, d3)  # (B, 512, 16, 16)
        u5 = self.up5(u4, d2)  # (B, 256, 32, 32)
        u6 = self.up6(u5, d1)  # (B, 128, 64, 64)
        return self.final(u6)   # (B,   3,128,128)


G = UNetGenerator().to(device)
print(f'Generator parameters: {sum(p.numel() for p in G.parameters()):,}')

# Verify output shape
dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(device)
print(f'Generator output shape: {G(dummy).shape}')  # Should be (1, 3, 128, 128)

In [ ]:
class PatchGANDiscriminator(nn.Module):
    """
    PatchGAN Discriminator.
    Input : concat(edge_map, real/fake_shoe) → 6 channels
    Output: patch map (not a single scalar)
    Each output pixel = real/fake decision for a 70x70 receptive field patch
    No fully connected layers.
    """
    def __init__(self, in_channels=6, features=[64, 128, 256, 512]):
        super().__init__()
        layers = []

        # First layer: no BN
        layers += [
            nn.Conv2d(in_channels, features[0], 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        ]

        # Intermediate layers
        in_ch = features[0]
        for out_ch in features[1:]:
            stride = 2 if out_ch != features[-1] else 1
            layers += [
                nn.Conv2d(in_ch, out_ch, 4, stride=stride, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.LeakyReLU(0.2, inplace=True)
            ]
            in_ch = out_ch

        # Final layer: output 1-channel patch map
        layers.append(nn.Conv2d(in_ch, 1, 4, stride=1, padding=1))

        self.model = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.0, 0.02)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight, 1.0, 0.02)
                nn.init.zeros_(m.bias)

    def forward(self, input_img, target_img):
        # Concatenate input and target along channel dim
        x = torch.cat([input_img, target_img], dim=1)  # (B, 6, 256, 256)
        return self.model(x)  # (B, 1, 30, 30) patch output


D = PatchGANDiscriminator().to(device)
d_params = sum(p.numel() for p in D.parameters())
print(f'Discriminator (PatchGAN) parameters: {d_params:,}')

# Show output shape
dummy_in  = torch.zeros(1, 3, 256, 256).to(device)
dummy_out = D(dummy_in, dummy_in)
print(f'PatchGAN output shape: {dummy_out.shape}')  # Expected: (1,1,30,30)

In [ ]:
# Loss functions
criterion_GAN = nn.BCEWithLogitsLoss()
criterion_L1  = nn.L1Loss()

# Optimizers (same LR for G and D as per Pix2Pix paper)
optimizer_G = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, BETA2))
optimizer_D = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, BETA2))

print('Generator Optimizer  : Adam (lr=0.0002, beta1=0.5)')
print('Discriminator Optimizer: Adam (lr=0.0002, beta1=0.5)')
print(f'Adversarial Loss     : BCEWithLogitsLoss')
print(f'Reconstruction Loss  : L1 (lambda={L1_LAMBDA})')

In [ ]:
from tqdm.notebook import tqdm

G_losses, D_losses, L1_losses = [], [], []

# Fix label shape for 128x128 input → PatchGAN output is 13x13
dummy    = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(device)
patch_h  = D(dummy, dummy).shape[2]
patch_w  = D(dummy, dummy).shape[3]
print(f'PatchGAN output patch size: {patch_h}x{patch_w}')

print(f'\nStarting Pix2Pix training for {NUM_EPOCHS} epochs...\n')

for epoch in range(1, NUM_EPOCHS + 1):
    G.train(); D.train()
    ep_G, ep_D, ep_L1 = 0.0, 0.0, 0.0

    loop = tqdm(train_loader, desc=f'Epoch [{epoch}/{NUM_EPOCHS}]', leave=False)

    for edges_img, real_img in loop:
        edges_img = edges_img.to(device)
        real_img  = real_img.to(device)
        batch_sz  = edges_img.size(0)

        # ── Dynamically sized labels ──────────────
        real_label = torch.ones (batch_sz, 1, patch_h, patch_w, device=device)
        fake_label = torch.zeros(batch_sz, 1, patch_h, patch_w, device=device)

        # ── Train Discriminator ───────────────────
        optimizer_D.zero_grad()
        pred_real    = D(edges_img, real_img)
        loss_D_real  = criterion_GAN(pred_real, real_label)

        fake_img     = G(edges_img).detach()
        pred_fake    = D(edges_img, fake_img)
        loss_D_fake  = criterion_GAN(pred_fake, fake_label)

        loss_D = (loss_D_real + loss_D_fake) * 0.5
        loss_D.backward()
        optimizer_D.step()

        # ── Train Generator ───────────────────────
        optimizer_G.zero_grad()
        fake_img     = G(edges_img)
        pred_fake    = D(edges_img, fake_img)
        loss_G_adv   = criterion_GAN(pred_fake, real_label)
        loss_G_L1    = criterion_L1(fake_img, real_img) * L1_LAMBDA
        loss_G       = loss_G_adv + loss_G_L1
        loss_G.backward()
        optimizer_G.step()

        ep_G  += loss_G_adv.item()
        ep_D  += loss_D.item()
        ep_L1 += loss_G_L1.item() / L1_LAMBDA

        loop.set_postfix(G=f'{loss_G_adv.item():.3f}',
                         D=f'{loss_D.item():.3f}',
                         L1=f'{loss_G_L1.item()/L1_LAMBDA:.3f}')

    avg_G  = ep_G  / len(train_loader)
    avg_D  = ep_D  / len(train_loader)
    avg_L1 = ep_L1 / len(train_loader)

    G_losses.append(avg_G)
    D_losses.append(avg_D)
    L1_losses.append(avg_L1)

    print(f'Epoch [{epoch:3d}/{NUM_EPOCHS}] | '
          f'G_adv: {avg_G:.4f} | D: {avg_D:.4f} | L1: {avg_L1:.4f}')

print('\nTraining complete!')

In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Pix2Pix Training Loss Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, G_losses, color='royalblue', linewidth=2)
axes[0].set_title('Generator Adversarial Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, D_losses, color='tomato', linewidth=2)
axes[1].set_title('Discriminator Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, L1_losses, color='green', linewidth=2)
axes[2].set_title('L1 Reconstruction Loss')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('L1')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
G.eval()
val_edges, val_reals = next(iter(val_loader))
val_edges_gpu = val_edges.to(device)

with torch.no_grad():
    val_fakes = G(val_edges_gpu).cpu()

n = 8
fig, axes = plt.subplots(3, n, figsize=(20, 7))
fig.suptitle('Pix2Pix Results\nRow 1: Edge Input | Row 2: Generated Output | Row 3: Real Target',
             fontsize=12, fontweight='bold')

row_data   = [val_edges, val_fakes, val_reals]
row_labels = ['Edge Input', 'Generated', 'Real Target']

for row_idx, (data, label) in enumerate(zip(row_data, row_labels)):
    for col_idx in range(n):
        img = denormalize(data[col_idx]).permute(1,2,0).numpy()
        axes[row_idx, col_idx].imshow(np.clip(img, 0, 1))
        axes[row_idx, col_idx].axis('off')
    axes[row_idx, 0].set_ylabel(label, fontsize=9, rotation=0, labelpad=60, va='center')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(10, 14))
fig.suptitle('Pix2Pix – Detailed 4-Sample Results', fontsize=13, fontweight='bold')

for col, title in enumerate(['Edge Input', 'Generated Output', 'Real Target']):
    axes[0, col].set_title(title, fontsize=10, fontweight='bold', color='navy')

for i in range(4):
    imgs = [val_edges[i], val_fakes[i], val_reals[i]]
    for j, img_t in enumerate(imgs):
        axes[i, j].imshow(np.clip(denormalize(img_t).permute(1,2,0).numpy(), 0, 1))
        axes[i, j].axis('off')

    mse_val = ((val_fakes[i] - val_reals[i])**2).mean().item()
    l1_val  = (val_fakes[i] - val_reals[i]).abs().mean().item()
    axes[i, 0].set_ylabel(f'Sample {i+1}', fontsize=9, rotation=0, labelpad=45, va='center')
    axes[i, 1].set_xlabel(f'MSE={mse_val:.4f} | L1={l1_val:.4f}', fontsize=8, color='green')

plt.tight_layout()
plt.show()

In [ ]:
# ── Baseline CNN Encoder-Decoder (same U-Net architecture, NO discriminator)
# We retrain a small version on the same data to compare fairly

class BaselineCNN(nn.Module):
    """
    Simple Encoder-Decoder (no adversarial training).
    Same U-Net structure as G but trained with MSE+L1 only.
    """
    def __init__(self):
        super().__init__()
        f = 64
        # Encoder
        self.e1 = nn.Sequential(nn.Conv2d(3, f,   4, 2, 1), nn.LeakyReLU(0.2))
        self.e2 = nn.Sequential(nn.Conv2d(f,   f*2, 4, 2, 1), nn.BatchNorm2d(f*2),   nn.LeakyReLU(0.2))
        self.e3 = nn.Sequential(nn.Conv2d(f*2, f*4, 4, 2, 1), nn.BatchNorm2d(f*4),   nn.LeakyReLU(0.2))
        self.e4 = nn.Sequential(nn.Conv2d(f*4, f*8, 4, 2, 1), nn.BatchNorm2d(f*8),   nn.LeakyReLU(0.2))
        self.bn = nn.Sequential(nn.Conv2d(f*8, f*8, 4, 2, 1), nn.ReLU())
        # Decoder
        self.d1 = nn.Sequential(nn.ConvTranspose2d(f*8,   f*8, 4, 2, 1), nn.BatchNorm2d(f*8), nn.ReLU(), nn.Dropout(0.5))
        self.d2 = nn.Sequential(nn.ConvTranspose2d(f*8*2, f*4, 4, 2, 1), nn.BatchNorm2d(f*4), nn.ReLU())
        self.d3 = nn.Sequential(nn.ConvTranspose2d(f*4*2, f*2, 4, 2, 1), nn.BatchNorm2d(f*2), nn.ReLU())
        self.d4 = nn.Sequential(nn.ConvTranspose2d(f*2*2, f,   4, 2, 1), nn.BatchNorm2d(f),   nn.ReLU())
        self.out= nn.Sequential(nn.ConvTranspose2d(f*2,   3,   4, 2, 1), nn.Tanh())

    def forward(self, x):
        e1 = self.e1(x);  e2 = self.e2(e1); e3 = self.e3(e2); e4 = self.e4(e3)
        bn = self.bn(e4)
        d1 = self.d1(bn)
        d2 = self.d2(torch.cat([d1, e4], 1))
        d3 = self.d3(torch.cat([d2, e3], 1))
        d4 = self.d4(torch.cat([d3, e2], 1))
        return self.out(torch.cat([d4, e1], 1))


baseline = BaselineCNN().to(device)
opt_b    = optim.Adam(baseline.parameters(), lr=LR, betas=(BETA1, BETA2))
loss_mse = nn.MSELoss(); loss_l1 = nn.L1Loss()

print('Training Baseline CNN (20 epochs for comparison)...')
baseline_losses = []

for epoch in range(1, 21):
    baseline.train()
    ep_loss = 0.0
    for edges_b, reals_b in train_loader:
        edges_b = edges_b.to(device); reals_b = reals_b.to(device)
        out  = baseline(edges_b)
        loss = loss_mse(out, reals_b) + 10.0 * loss_l1(out, reals_b)
        opt_b.zero_grad(); loss.backward(); opt_b.step()
        ep_loss += loss.item()
    avg = ep_loss / len(train_loader)
    baseline_losses.append(avg)
    if epoch % 5 == 0:
        print(f'  Baseline Epoch [{epoch}/20] Loss: {avg:.4f}')

print('Baseline training done!')

In [ ]:
baseline.eval(); G.eval()
val_edges_b, val_reals_b = next(iter(val_loader))
val_edges_gpu = val_edges_b.to(device)

with torch.no_grad():
    baseline_out = baseline(val_edges_gpu).cpu()
    pix2pix_out  = G(val_edges_gpu).cpu()

n = 6
fig, axes = plt.subplots(4, n, figsize=(18, 10))
fig.suptitle('Task 5: Pix2Pix GAN vs Baseline CNN Encoder-Decoder',
             fontsize=14, fontweight='bold')

row_data   = [val_edges_b, baseline_out, pix2pix_out, val_reals_b]
row_labels = ['Input\n(Edge)', 'Baseline\nCNN', 'Pix2Pix\nGAN', 'Target\n(Real)']
row_colors = ['black', 'darkorange', 'royalblue', 'green']

for row_idx, (data, label, color) in enumerate(zip(row_data, row_labels, row_colors)):
    for col_idx in range(n):
        img = np.clip(denormalize(data[col_idx]).permute(1,2,0).numpy(), 0, 1)
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].axis('off')
    axes[row_idx, 0].set_ylabel(label, fontsize=10, color=color,
                                 rotation=0, labelpad=55, va='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Pix2Pix metrics ──────────────────────────────────────
G.eval()
all_mse_pix, all_l1_pix = [], []

with torch.no_grad():
    for edges_b, reals_b in val_loader:
        edges_b = edges_b.to(device)
        reals_b = reals_b.to(device)
        fakes_b = G(edges_b)
        for i in range(edges_b.size(0)):
            all_mse_pix.append(((fakes_b[i] - reals_b[i])**2).mean().item())
            all_l1_pix.append((fakes_b[i] - reals_b[i]).abs().mean().item())

pix_mse  = np.mean(all_mse_pix)
pix_l1   = np.mean(all_l1_pix)
pix_psnr = 10 * np.log10(4.0 / pix_mse)

# ── Baseline metrics ─────────────────────────────────────
baseline.eval()
all_mse_base, all_l1_base = [], []

with torch.no_grad():
    for edges_b, reals_b in val_loader:
        edges_b = edges_b.to(device)
        reals_b = reals_b.to(device)
        out = baseline(edges_b)
        for i in range(edges_b.size(0)):
            all_mse_base.append(((out[i] - reals_b[i])**2).mean().item())
            all_l1_base.append((out[i] - reals_b[i]).abs().mean().item())

base_mse  = np.mean(all_mse_base)
base_l1   = np.mean(all_l1_base)
base_psnr = 10 * np.log10(4.0 / base_mse)

# ── Print table ──────────────────────────────────────────
print('=' * 55)
print(f'{"Metric":<15} {"Baseline CNN":>18} {"Pix2Pix GAN":>18}')
print('=' * 55)
print(f'{"MSE":<15} {base_mse:>18.5f} {pix_mse:>18.5f}')
print(f'{"L1":<15} {base_l1:>18.5f} {pix_l1:>18.5f}')
print(f'{"PSNR (dB)":<15} {base_psnr:>18.2f} {pix_psnr:>18.2f}')
print('=' * 55)
winner = 'Pix2Pix GAN' if pix_psnr > base_psnr else 'Baseline CNN'
print(f'  Better PSNR → {winner}')
print('=' * 55)

# ── Bar chart ────────────────────────────────────────────
metrics       = ['MSE ↓', 'L1 ↓', 'PSNR ↑ (dB)']
baseline_vals = [base_mse, base_l1, base_psnr]
pix2pix_vals  = [pix_mse,  pix_l1,  pix_psnr]

x     = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline CNN', color='darkorange', alpha=0.85)
b2 = ax.bar(x + width/2, pix2pix_vals,  width, label='Pix2Pix GAN', color='royalblue',  alpha=0.85)

ax.set_title('Pix2Pix GAN vs Baseline CNN – Metric Comparison',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()